# Checkpoint 7 Data Cleaning

Sébastien Vouters - 06/26/2026

Run the setup cell first. Complete Part 1 and then Part 2 if time allows.


In [1]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("output/checkpoint")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The source DataFrames are defined row-wise so the data is easier to read.
# Each inner list is one source row.
# The column list below each row list defines what each value means.

warehouse_shipments = pd.DataFrame(
    [
        ["WH001", " s-1001 ", " east ",   "2026-06-24", "12.5", "packed",  "2026-06-24 09:00"],
        ["WH002", "S-1002",   "West",     "2026-06-24", "8.0",  "shipped", "2026-06-24 10:00"],
        ["WH003", "S-1003",   "North",    "2026-06-25", "25.0", "shipped", "2026-06-25 08:30"],
        ["WH004", "",         "South",    "2026-06-25", "5.0",  "packed",  "2026-06-25 09:00"],
        ["WH005", "S-1004",   "Central",  "2026-06-25", "bad",  "packed",  "2026-06-25 09:20"],
        ["WH006", "S-1005",   "East",     "bad-date",   "7.5",  "packed",  "2026-06-25 10:00"],
        ["WH007", "S-1006",   "West",     "2026-06-25", "10.0", "packed",  "bad-date"],
        ["WH008", "S-1007",   "North",    "2026-06-26", "18.0", "shipped", "2026-06-26 09:00"],
        ["WH009", "S-1008",   "Mars",     "2026-06-26", "6.0",  "packed",  "2026-06-26 09:10"],
        ["WH010", "S-1014",   "Central",  "2026-06-26", "22.0", "shipped", "2026-06-26 07:00"],
    ],
    columns=[
        "wh_shipment_id",
        "shipment_ref",
        "zone",
        "ship_date",
        "weight_kg",
        "status",
        "updated_at",
    ],
)

carrier_shipments = pd.DataFrame(
    [
        ["CR501", "S-1001",   "East",    "2026-06-24", "13.0 kg", "in_transit", "2026-06-24 11:00"],
        ["CR502", " s-1002 ", "West",    "2026-06-24", "8.2",     "in_transit", "2026-06-24 10:00"],
        ["CR503", "S-1007",   "North",   "2026-06-26", "19.5 kg", "in_transit", "2026-06-26 08:00"],
        ["CR504", "S-1014",   "Central", "2026-06-26", "21.0 kg", "delivered",  "2026-06-26 08:00"],
        ["CR505", "S-1015",   "South",   "2026-06-27", "9.0 kg",  "delivered",  "2026-06-27 09:30"],
        ["CR506", "S-1009",   "South",   "2026-06-26", "",        "new",        "2026-06-26 11:00"],
        ["CR507", "S-1010",   "North",   "",           "11.0 kg", "new",        "2026-06-26 11:20"],
        ["CR508", "S-1011",   "East",    "2026-06-26", "7.0 kg",  "new",        "not-a-date"],
        ["CR509", "S-1012",   "Unknown", "2026-06-26", "4.0 kg",  "new",        "2026-06-26 11:40"],
    ],
    columns=[
        "carrier_record_id",
        "tracking_ref",
        "service_zone",
        "planned_date",
        "weight_text",
        "carrier_state",
        "last_update",
    ],
)

norway_zone_lookup = pd.DataFrame(
    [
        ["East",    "Eastern Norway"],
        ["West",    "Western Norway"],
        ["North",   "Northern Norway"],
        ["South",   "Southern Norway"],
        ["Central", "Central Norway"],
    ],
    columns=[
        "zone",
        "region",
    ],
)



## Part 1 - Required tasks


In [2]:
# Task 1 - Inspect input DataFrames
names = ['Warehouse shipments', 'Carrier shipments', 'Norway zone lookup']
tables =[warehouse_shipments, carrier_shipments, norway_zone_lookup]

for name, table in zip(names, tables) :
    print('----------------------------')
    print(name, 'number of rows :', len(table))
    display(table)


----------------------------
Warehouse shipments number of rows : 10


,wh_shipment_id,shipment_ref,zone,ship_date,weight_kg,status,updated_at
0,WH001,s-1001,east,2026-06-24,12.5,packed,2026-06-24 09:00
1,WH002,S-1002,West,2026-06-24,8.0,shipped,2026-06-24 10:00
2,WH003,S-1003,North,2026-06-25,25.0,shipped,2026-06-25 08:30
3,WH004,,South,2026-06-25,5.0,packed,2026-06-25 09:00
4,WH005,S-1004,Central,2026-06-25,bad,packed,2026-06-25 09:20
5,WH006,S-1005,East,bad-date,7.5,packed,2026-06-25 10:00
6,WH007,S-1006,West,2026-06-25,10.0,packed,bad-date
7,WH008,S-1007,North,2026-06-26,18.0,shipped,2026-06-26 09:00
8,WH009,S-1008,Mars,2026-06-26,6.0,packed,2026-06-26 09:10
9,WH010,S-1014,Central,2026-06-26,22.0,shipped,2026-06-26 07:00


----------------------------
Carrier shipments number of rows : 9


,carrier_record_id,tracking_ref,service_zone,planned_date,weight_text,carrier_state,last_update
0,CR501,S-1001,East,2026-06-24,13.0 kg,in_transit,2026-06-24 11:00
1,CR502,s-1002,West,2026-06-24,8.2,in_transit,2026-06-24 10:00
2,CR503,S-1007,North,2026-06-26,19.5 kg,in_transit,2026-06-26 08:00
3,CR504,S-1014,Central,2026-06-26,21.0 kg,delivered,2026-06-26 08:00
4,CR505,S-1015,South,2026-06-27,9.0 kg,delivered,2026-06-27 09:30
5,CR506,S-1009,South,2026-06-26,,new,2026-06-26 11:00
6,CR507,S-1010,North,,11.0 kg,new,2026-06-26 11:20
7,CR508,S-1011,East,2026-06-26,7.0 kg,new,not-a-date
8,CR509,S-1012,Unknown,2026-06-26,4.0 kg,new,2026-06-26 11:40


----------------------------
Norway zone lookup number of rows : 5


,zone,region
0,East,Eastern Norway
1,West,Western Norway
2,North,Northern Norway
3,South,Southern Norway
4,Central,Central Norway


In [3]:
# Task 2 - Standardize warehouse_shipments
warehouse_std = pd.DataFrame({
        "source_system": "warehouse",
        "source_record_id": warehouse_shipments["wh_shipment_id"],
        "source_priority": 2,
        "shipment_ref": warehouse_shipments["shipment_ref"],
        "zone": warehouse_shipments["zone"],
        "ship_date": warehouse_shipments["ship_date"],
        "weight_kg": warehouse_shipments["weight_kg"],
        "status": warehouse_shipments['status'],
        "updated_at": warehouse_shipments['updated_at'],
    })

warehouse_std

,source_system,source_record_id,source_priority,shipment_ref,zone,ship_date,weight_kg,status,updated_at
0,warehouse,WH001,2,s-1001,east,2026-06-24,12.5,packed,2026-06-24 09:00
1,warehouse,WH002,2,S-1002,West,2026-06-24,8.0,shipped,2026-06-24 10:00
2,warehouse,WH003,2,S-1003,North,2026-06-25,25.0,shipped,2026-06-25 08:30
3,warehouse,WH004,2,,South,2026-06-25,5.0,packed,2026-06-25 09:00
4,warehouse,WH005,2,S-1004,Central,2026-06-25,bad,packed,2026-06-25 09:20
5,warehouse,WH006,2,S-1005,East,bad-date,7.5,packed,2026-06-25 10:00
6,warehouse,WH007,2,S-1006,West,2026-06-25,10.0,packed,bad-date
7,warehouse,WH008,2,S-1007,North,2026-06-26,18.0,shipped,2026-06-26 09:00
8,warehouse,WH009,2,S-1008,Mars,2026-06-26,6.0,packed,2026-06-26 09:10
9,warehouse,WH010,2,S-1014,Central,2026-06-26,22.0,shipped,2026-06-26 07:00


In [4]:
# Task 3 - Standardize carrier_shipments
carrier_std = pd.DataFrame({
        "source_system": "carrier",
        "source_record_id": carrier_shipments["carrier_record_id"],
        "source_priority": 1,
        "shipment_ref": carrier_shipments["tracking_ref"],
        "zone": carrier_shipments["service_zone"],
        "ship_date": carrier_shipments["planned_date"],
        "weight_kg": carrier_shipments["weight_text"],
        "status": carrier_shipments['carrier_state'],
        "updated_at": carrier_shipments['last_update'],
    })

carrier_std

,source_system,source_record_id,source_priority,shipment_ref,zone,ship_date,weight_kg,status,updated_at
0,carrier,CR501,1,S-1001,East,2026-06-24,13.0 kg,in_transit,2026-06-24 11:00
1,carrier,CR502,1,s-1002,West,2026-06-24,8.2,in_transit,2026-06-24 10:00
2,carrier,CR503,1,S-1007,North,2026-06-26,19.5 kg,in_transit,2026-06-26 08:00
3,carrier,CR504,1,S-1014,Central,2026-06-26,21.0 kg,delivered,2026-06-26 08:00
4,carrier,CR505,1,S-1015,South,2026-06-27,9.0 kg,delivered,2026-06-27 09:30
5,carrier,CR506,1,S-1009,South,2026-06-26,,new,2026-06-26 11:00
6,carrier,CR507,1,S-1010,North,,11.0 kg,new,2026-06-26 11:20
7,carrier,CR508,1,S-1011,East,2026-06-26,7.0 kg,new,not-a-date
8,carrier,CR509,1,S-1012,Unknown,2026-06-26,4.0 kg,new,2026-06-26 11:40


In [5]:
# Task 4 - Combine and clean
shipments_all = pd.concat([warehouse_std, carrier_std], ignore_index=True)
print(' Shipments combined number of rows :', len(shipments_all))

shipments_all ["shipment_ref_clean"] = shipments_all["shipment_ref"].astype("string").str.strip().str.upper().replace("", pd.NA)
shipments_all["zone_clean"] = shipments_all["zone"].astype("string").str.strip().str.title().replace("", pd.NA).replace("Unknown", pd.NA)
shipments_all["weight_kg_clean"] = pd.to_numeric(
    shipments_all["weight_kg"].astype("string").str.replace(" kg", "").str.strip().replace("", pd.NA),
    errors="coerce"
)
shipments_all["ship_date_clean"] = pd.to_datetime(shipments_all["ship_date"], errors="coerce")
shipments_all['updated_at_clean'] = pd.to_datetime(shipments_all["updated_at"], errors="coerce")

shipments_all

 Shipments combined number of rows : 19


,source_system,source_record_id,source_priority,shipment_ref,zone,ship_date,weight_kg,status,updated_at,shipment_ref_clean,zone_clean,weight_kg_clean,ship_date_clean,updated_at_clean
0,warehouse,WH001,2,s-1001,east,2026-06-24,12.5,packed,2026-06-24 09:00,S-1001,East,12.5,2026-06-24,2026-06-24 09:00:00
1,warehouse,WH002,2,S-1002,West,2026-06-24,8.0,shipped,2026-06-24 10:00,S-1002,West,8.0,2026-06-24,2026-06-24 10:00:00
2,warehouse,WH003,2,S-1003,North,2026-06-25,25.0,shipped,2026-06-25 08:30,S-1003,North,25.0,2026-06-25,2026-06-25 08:30:00
3,warehouse,WH004,2,,South,2026-06-25,5.0,packed,2026-06-25 09:00,<NA>,South,5.0,2026-06-25,2026-06-25 09:00:00
4,warehouse,WH005,2,S-1004,Central,2026-06-25,bad,packed,2026-06-25 09:20,S-1004,Central,<NA>,2026-06-25,2026-06-25 09:20:00
5,warehouse,WH006,2,S-1005,East,bad-date,7.5,packed,2026-06-25 10:00,S-1005,East,7.5,NaT,2026-06-25 10:00:00
6,warehouse,WH007,2,S-1006,West,2026-06-25,10.0,packed,bad-date,S-1006,West,10.0,2026-06-25,NaT
7,warehouse,WH008,2,S-1007,North,2026-06-26,18.0,shipped,2026-06-26 09:00,S-1007,North,18.0,2026-06-26,2026-06-26 09:00:00
8,warehouse,WH009,2,S-1008,Mars,2026-06-26,6.0,packed,2026-06-26 09:10,S-1008,Mars,6.0,2026-06-26,2026-06-26 09:10:00
9,warehouse,WH010,2,S-1014,Central,2026-06-26,22.0,shipped,2026-06-26 07:00,S-1014,Central,22.0,2026-06-26,2026-06-26 07:00:00


In [6]:
# Task 5 - Join zone lookup and create review_reason
shipments_enriched = shipments_all.merge(
    norway_zone_lookup,
    left_on="zone_clean",
    right_on="zone",
    how="left"
)
print('Shipment enriched')
display(shipments_enriched[["source_record_id", "zone_clean", "region"]].head())

# review_reason creation
shipments_enriched["review_reason"] = pd.NA

shipments_enriched.loc[
    shipments_enriched["review_reason"].isna() & shipments_enriched["shipment_ref_clean"].isna(),
    "review_reason"
] = "MISSING_SHIPMENT_REF"

shipments_enriched.loc[
    shipments_enriched["review_reason"].isna() & shipments_enriched["weight_kg_clean"].isna(),
    "review_reason"
] = "INVALID_WEIGHT"

shipments_enriched.loc[
    shipments_enriched["review_reason"].isna() & shipments_enriched["ship_date_clean"].isna(),
    "review_reason"
] = "INVALID_SHIP_DATE"

shipments_enriched.loc[
    shipments_enriched["review_reason"].isna() & shipments_enriched["updated_at_clean"].isna(),
    "review_reason"
] = "INVALID_UPDATED_AT"

shipments_enriched.loc[
    shipments_enriched["review_reason"].isna() & shipments_enriched["region"].isna(),
    "review_reason"
] = "UNKNOWN_ZONE"

print('Invalid rows')
shipments_enriched[~shipments_enriched["review_reason"].isna()][["source_record_id", "review_reason"]]


Shipment enriched


,source_record_id,zone_clean,region
0,WH001,East,Eastern Norway
1,WH002,West,Western Norway
2,WH003,North,Northern Norway
3,WH004,South,Southern Norway
4,WH005,Central,Central Norway


Invalid rows


,source_record_id,review_reason
3,WH004,MISSING_SHIPMENT_REF
4,WH005,INVALID_WEIGHT
5,WH006,INVALID_SHIP_DATE
6,WH007,INVALID_UPDATED_AT
8,WH009,UNKNOWN_ZONE
15,CR506,INVALID_WEIGHT
16,CR507,INVALID_SHIP_DATE
17,CR508,INVALID_UPDATED_AT
18,CR509,UNKNOWN_ZONE


In [7]:
# Task 6 - Split valid_shipments and shipment_review_rows
valid_shipments = shipments_enriched[shipments_enriched["review_reason"].isna()]
shipment_review_rows = shipments_enriched[~shipments_enriched["review_reason"].isna()]

print(len(valid_shipments), 'valid rows')
print(len(shipment_review_rows), 'rejected rows')

10 valid rows
9 rejected rows


In [8]:
# Task 7 - Create golden_shipments
golden_shipments = (
    valid_shipments
    .sort_values(
        ["shipment_ref_clean", "updated_at_clean", "source_priority", "weight_kg_clean"],
        ascending=[True, False, True, False]
    )
    .drop_duplicates(subset=["shipment_ref_clean"], keep="first")
    .reset_index(drop=True)
)[[
    'source_system',
    'source_record_id',
    'shipment_ref_clean',
    'zone_clean',
    'region',
    'ship_date_clean',
    'weight_kg_clean',
    'status',
    'updated_at_clean'
]]

print(len(golden_shipments), 'golden shipments rows')
golden_shipments

6 golden shipments rows


,source_system,source_record_id,shipment_ref_clean,zone_clean,region,ship_date_clean,weight_kg_clean,status,updated_at_clean
0,carrier,CR501,S-1001,East,Eastern Norway,2026-06-24,13.0,in_transit,2026-06-24 11:00:00
1,carrier,CR502,S-1002,West,Western Norway,2026-06-24,8.2,in_transit,2026-06-24 10:00:00
2,warehouse,WH003,S-1003,North,Northern Norway,2026-06-25,25.0,shipped,2026-06-25 08:30:00
3,warehouse,WH008,S-1007,North,Northern Norway,2026-06-26,18.0,shipped,2026-06-26 09:00:00
4,carrier,CR504,S-1014,Central,Central Norway,2026-06-26,21.0,delivered,2026-06-26 08:00:00
5,carrier,CR505,S-1015,South,Southern Norway,2026-06-27,9.0,delivered,2026-06-27 09:30:00


In [9]:
# Task 8 - Create summary and write outputs
shipment_summary = pd.DataFrame([
    {"metric_name": "warehouse_source_rows", "metric_value": len(warehouse_shipments)},
    {"metric_name": "carrier_source_rows", "metric_value": len(carrier_shipments)},
    {"metric_name": "combined_rows", "metric_value": len(shipments_all)},
    {"metric_name": "valid_shipments_rows", "metric_value": len(valid_shipments)},
    {"metric_name": "review_rows", "metric_value": len(shipment_review_rows)},
    {"metric_name": "golden_shipment_rows", "metric_value": len(golden_shipments)},
    {"metric_name": "unknown_zone_rows", "metric_value": len(shipment_review_rows[shipment_review_rows["review_reason"]=="UNKNOWN_ZONE"])},
])

print('Cleaning summary')
display(shipment_summary)

# Writing files
dfs = [shipments_all, valid_shipments, shipment_review_rows, golden_shipments, shipment_summary]
file_names = ['shipments_all.csv', 'valid_shipments.csv', 'shipment_review_rows.csv', 'golden_shipments.csv', 'shipment_summary.csv']

def write_csv(df, file_name, dir = OUTPUT_DIR, use_index=False) :
    print('Writing', dir / file_name)
    try :
        df.to_csv(
            dir / file_name,
            index=use_index
        )
        print(len(df), 'row written')
    except :
        print("An error occured")

for df, file_name in zip(dfs, file_names) :
    write_csv(df, file_name)


Cleaning summary


,metric_name,metric_value
0,warehouse_source_rows,10
1,carrier_source_rows,9
2,combined_rows,19
3,valid_shipments_rows,10
4,review_rows,9
5,golden_shipment_rows,6
6,unknown_zone_rows,2


Writing output/checkpoint/shipments_all.csv
19 row written
Writing output/checkpoint/valid_shipments.csv
10 row written
Writing output/checkpoint/shipment_review_rows.csv
9 row written
Writing output/checkpoint/golden_shipments.csv
6 row written
Writing output/checkpoint/shipment_summary.csv
7 row written


## Part 2 - Compensation / advanced tasks


In [10]:
# Task 9 - Duplicate shipment audit
duplicate_shipment_audit = valid_shipments.groupby('shipment_ref_clean').agg(
    source_row_count = ("shipment_ref_clean", "count"),
    newest_updated_at = ("updated_at_clean", "max"),
    max_weight_kg = ('weight_kg_clean', 'max')
    )

duplicate_shipment_audit = duplicate_shipment_audit[duplicate_shipment_audit['source_row_count'] > 1]

display(duplicate_shipment_audit)
write_csv(duplicate_shipment_audit, 'duplicate_shipment_audit.csv')

,source_row_count,newest_updated_at,max_weight_kg
shipment_ref_clean,,,
S-1001,2,2026-06-24 11:00:00,13.0
S-1002,2,2026-06-24 10:00:00,8.2
S-1007,2,2026-06-26 09:00:00,19.5
S-1014,2,2026-06-26 08:00:00,22.0


Writing output/checkpoint/duplicate_shipment_audit.csv
4 row written


In [11]:
# Task 10 - Review reason summary
review_reason_summary = shipment_review_rows.groupby('review_reason').agg(
    review_count=("review_reason", "count")
)

display(review_reason_summary)
write_csv(review_reason_summary, 'review_reason_summary.csv', use_index=True)

,review_count
review_reason,
INVALID_SHIP_DATE,2
INVALID_UPDATED_AT,2
INVALID_WEIGHT,2
MISSING_SHIPMENT_REF,1
UNKNOWN_ZONE,2


Writing output/checkpoint/review_reason_summary.csv
5 row written


In [12]:
# Task 11 - Region summary
region_summary = golden_shipments.groupby('region').agg(
    shipment_count = ("shipment_ref_clean", "count"),
    total_weight_kg = ("weight_kg_clean", "sum"),
    average_weight_kg = ('weight_kg_clean', 'mean')
    )

display(region_summary)
write_csv(region_summary, 'region_summary.csv')

,shipment_count,total_weight_kg,average_weight_kg
region,,,
Central Norway,1,21.0,21.0
Eastern Norway,1,13.0,13.0
Northern Norway,2,43.0,21.5
Southern Norway,1,9.0,9.0
Western Norway,1,8.2,8.2


Writing output/checkpoint/region_summary.csv
5 row written


In [13]:
# Task 12 - Heavy shipments
heavy_shipments = golden_shipments[golden_shipments['weight_kg_clean'] >= 20]

display(heavy_shipments)
write_csv(heavy_shipments, 'heavy_shipments.csv')

,source_system,source_record_id,shipment_ref_clean,zone_clean,region,ship_date_clean,weight_kg_clean,status,updated_at_clean
2,warehouse,WH003,S-1003,North,Northern Norway,2026-06-25,25.0,shipped,2026-06-25 08:30:00
4,carrier,CR504,S-1014,Central,Central Norway,2026-06-26,21.0,delivered,2026-06-26 08:00:00


Writing output/checkpoint/heavy_shipments.csv
2 row written


In [14]:
# Task 13 - Add delivery_class
golden_shipments_with_delivery_class = golden_shipments.copy()
golden_shipments_with_delivery_class['delivery_class'] = golden_shipments_with_delivery_class['weight_kg_clean'].apply(
    lambda weight: "Small" if weight < 10 else ("Medium" if weight < 20 else "Large")
)

display(golden_shipments_with_delivery_class)
write_csv(golden_shipments_with_delivery_class, 'golden_shipments_with_delivery_class.csv')


,source_system,source_record_id,shipment_ref_clean,zone_clean,region,ship_date_clean,weight_kg_clean,status,updated_at_clean,delivery_class
0,carrier,CR501,S-1001,East,Eastern Norway,2026-06-24,13.0,in_transit,2026-06-24 11:00:00,Medium
1,carrier,CR502,S-1002,West,Western Norway,2026-06-24,8.2,in_transit,2026-06-24 10:00:00,Small
2,warehouse,WH003,S-1003,North,Northern Norway,2026-06-25,25.0,shipped,2026-06-25 08:30:00,Large
3,warehouse,WH008,S-1007,North,Northern Norway,2026-06-26,18.0,shipped,2026-06-26 09:00:00,Medium
4,carrier,CR504,S-1014,Central,Central Norway,2026-06-26,21.0,delivered,2026-06-26 08:00:00,Large
5,carrier,CR505,S-1015,South,Southern Norway,2026-06-27,9.0,delivered,2026-06-27 09:30:00,Small


Writing output/checkpoint/golden_shipments_with_delivery_class.csv
6 row written
